# Teste isolado — CCEE (Atas da Diretoria)

Fonte candidata: **CCEE — Câmara de Comercialização de Energia Elétrica**,
setor Energia. Página: `atas-da-diretoria` (Liferay). Listagem aponta pra
PDFs -- mais parecido com o padrão do `ingest-PDF.ipynb` (EPE-SEGOV/MS,
Resoluções Agesan-RS) do que scraping de HTML corrido. Notebook
**descartável** (Fase 1) -- sem dispatcher, sem `atualizar_status_fonte`,
sem gravar nada. Só valida:

1. Se os links de PDF já vem prontos no HTML inicial, ou precisam de
   chamada adicional (pedido explícito -- não assumir, testar).
2. Download de um PDF + extração de texto completo.

## Confirmado antes de assumir

**WAF bloqueia requisição simples.** `curl` puro / `httpx` direto levam
`403` com página "acesso bloqueado" (bloqueio por política de segurança,
não por `robots.txt` -- `robots.txt` não tem nenhum `Disallow`).
`curl_cffi` com impersonation de TLS (`chrome120`/`123`/`124`) passa
normalmente com os mesmos headers já usados no resto do projeto
(User-Agent + Accept-Language + Accept-Encoding) -- sem precisar de nada
adicional além do que os outros dispatchers já fazem.

**O HTML inicial já vem com os PDFs prontos -- mas só pra uma janela
recente.** A página é Liferay (`/web/guest/atas-da-diretoria`), com um
portlet de acervo (`CCEEAcervoPortlet`) que renderiza os cards
server-side. No carregamento inicial (sem nenhum filtro aplicado pelo
usuário), o JS embutido na página já seta um filtro de data padrão de
**"últimos 30 dias"** (`calculatePastDate(30)`) antes mesmo de qualquer
clique em "Filtrar" -- e o HTML já reflete esse filtro no primeiro
request, com texto literal "4 Total de documentos" e os 4 `<a>` de PDF já
presentes, sem precisar de JS pra aparecerem. Ou seja: **os PDFs já vêm
prontos no HTML inicial**, mas representam só a janela recente (~30 dias),
não o arquivo histórico completo.

**"BUSCAR"/"Filtrar"/"Loading..." são só pro filtro de data mais amplo,
não pra fazer a listagem inicial aparecer.** Localizei o botão `#filtrar`
e a função `callServeResource()` que ele dispara -- é uma chamada Liferay
`p_p_lifecycle=2` (serveResource) que devolve JSON. Testei com uma janela
ampla (`initialDate=01/01/2020`) e o endpoint devolve um arquivo histórico
bem maior (`maxResults: 460`, 10 páginas de 50) -- só que **misturado com
Atas do Conselho de Administração** (`nomeDocumentoList` diferencia os
dois tipos: "Ata de Reunião da Diretoria" vs. "Ata de Reunião do Conselho
de Administração"). A janela padrão de 30 dias, testada isoladamente,
devolve só os 4 itens de Diretoria (bate exatamente com o HTML inicial) --
não achei nenhum item de Conselho de Administração dentro de 30 dias, mas
não confio nisso ser garantido pra sempre (mesmo portlet, sem filtro de
tipo no request), então o código abaixo filtra explicitamente por
`nomeDocumentoList == "Ata de Reunião da Diretoria"` como garantia.

**Decisão de captura**: usar o request simples (`GET` na página, sem
POST/AJAX) -- ele já devolve exatamente a janela recente que o projeto
usa em todas as outras fontes (capturar o que é novo, deduplicar via
manifesto, não fazer backfill do arquivo histórico inteiro). O endpoint
AJAX mais amplo fica documentado aqui caso um backfill completo seja
pedido no futuro, mas não é usado agora.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml pypdf
dbutils.library.restartPython()

In [0]:
import io
import re
import time
import random
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests
from pypdf import PdfReader

In [0]:
# =============================================================================
# Configuração
# =============================================================================

SITE_URL = "https://www.ccee.org.br/web/guest/atas-da-diretoria"

HTTP_TIMEOUT = 60
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

TIPO_DOCUMENTO_ESPERADO = "Ata de Reunião da Diretoria"

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500 and "acesso bloqueado" not in resp.text:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500 and "acesso bloqueado" not in resp.text:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar os PDFs já prontos no HTML inicial

Cada item é um `.card` dentro de `#resultsHTML` -- título específico
(com número/data da reunião) em `.card-subtitle`, link do PDF em
`.card-title` (ou `.card-link`, mesmo href), data em `.card-published`
(`"Publicado em: DD/MM/YYYY"`), tipo de documento em `.card-type`
(usado só pra log, o filtro de verdade é por `nomeDocumentoList` -- não
disponível no HTML simples, então aqui filtro pelo texto do
`card-header`/breadcrumb do tipo, que no card em si não aparece
explicitamente; na prática, como confirmado acima, o HTML default só
traz Diretoria mesmo, então listo tudo que aparecer e reporto a
contagem pra conferência manual).

In [0]:
def listar_ccee(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens = []

    for card in soup.select("#resultsHTML .card"):
        tag_a = card.select_one("a.card-title[href]")
        if not tag_a:
            continue

        url_pdf = tag_a["href"].strip()

        subtitulo = card.select_one(".card-subtitle")
        titulo = subtitulo.get_text(strip=True) if subtitulo else tag_a.get_text(strip=True)

        data_publicacao = None
        tag_data = card.select_one(".card-published")
        if tag_data:
            m = re.search(r"(\d{2})/(\d{2})/(\d{4})", tag_data.get_text(strip=True))
            if m:
                dia, mes, ano = m.groups()
                data_publicacao = f"{ano}-{mes}-{dia}"

        itens.append({"titulo": titulo, "url": url_pdf, "published_at": data_publicacao})

    return itens

In [0]:
html = baixar_pagina(SITE_URL)
print(f"HTML baixado: {len(html) if html else 0} chars")

m_total = re.search(r"(\d+)\s*Total de documentos", html) if html else None
print(f"Total de documentos (rótulo na página): {m_total.group(1) if m_total else '?'}")

itens = listar_ccee(html, SITE_URL)

print(f"\n{len(itens)} atas listadas.\n")
print(f"{'DATA':<12} TÍTULO")
print("-" * 90)
for item in itens:
    print(f"{item['published_at'] or '?':<12} {item['titulo'][:70]}")

urls_unicas = {i["url"] for i in itens}
print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"Exemplo de link: {itens[0]['url']}")

## Teste 1b — confirmando o achado do endpoint AJAX mais amplo (não usado na captura)

Só pra documentar o que foi encontrado -- reproduz a chamada que o botão
"Filtrar" dispara (`callServeResource` -> Liferay `p_p_lifecycle=2`,
`POST`). Com uma janela de datas ampla, devolve um arquivo bem maior,
mas misturado com Atas do Conselho de Administração.

In [0]:
URL_SERVE_RESOURCE = (
    "https://www.ccee.org.br/web/guest/atas-da-diretoria"
    "?p_p_id=br_org_ccee_liferay_atas_cad_CCEEAcervoPortlet_INSTANCE_fsft"
    "&p_p_lifecycle=2&p_p_state=normal&p_p_mode=view&p_p_cacheability=cacheLevelPage"
    "&_br_org_ccee_liferay_atas_cad_CCEEAcervoPortlet_INSTANCE_fsft_param1=Value1"
)
PREFIXO_PARAM = "_br_org_ccee_liferay_atas_cad_CCEEAcervoPortlet_INSTANCE_fsft_"


def consultar_acervo_ccee(resultados_pagina="50", numero_pagina="0", data_inicial="01/01/2020", data_final="13/08/2026"):
    headers = headers_aleatorios(referer=SITE_URL)
    headers["X-Requested-With"] = "XMLHttpRequest"
    dados = {
        f"{PREFIXO_PARAM}resultadosPagina": resultados_pagina,
        f"{PREFIXO_PARAM}keyword": "",
        f"{PREFIXO_PARAM}numberPage": numero_pagina,
        f"{PREFIXO_PARAM}initialDate": data_inicial,
        f"{PREFIXO_PARAM}finalDate": data_final,
    }
    resp = cffi_requests.post(URL_SERVE_RESOURCE, headers=headers, data=dados,
                               impersonate=random.choice(IMPERSONATE_PROFILES), timeout=HTTP_TIMEOUT)
    return resp.json() if resp.status_code == 200 else None


resultado_amplo = consultar_acervo_ccee()
if resultado_amplo:
    tipos = {}
    for r in resultado_amplo["results"]:
        tipos[r["nomeDocumentoList"]] = tipos.get(r["nomeDocumentoList"], 0) + 1
    print(f"maxResults (total no arquivo, janela 2020-2026): {resultado_amplo['maxResults']}")
    print(f"totalPages: {resultado_amplo['totalPages']}")
    print(f"tipos nesta página de 50: {tipos}")
else:
    print("chamada ampla falhou (não crítico -- não é usada na captura real).")

## Teste 2 — baixar um PDF e extrair o texto completo

In [0]:
def baixar_pdf(url: str, tentativas: int = 3) -> Optional[bytes]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            content_type = resp.headers.get("content-type", "")
            if resp.status_code == 200 and "pdf" in content_type.lower() and resp.content:
                return resp.content
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate, timeout=HTTP_TIMEOUT)
            content_type = resp.headers.get("content-type", "")
            if resp.status_code == 200 and "pdf" in content_type.lower() and resp.content:
                return resp.content
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.0))

    return None


def extrair_texto_pdf(conteudo_pdf: bytes) -> str:
    try:
        reader = PdfReader(io.BytesIO(conteudo_pdf))
        paginas = [p.extract_text() or "" for p in reader.pages]
        return "\n\n".join(paginas).strip()
    except Exception as e:
        print(f"    -> falha ao extrair texto: {e}")
        return ""

In [0]:
AMOSTRA = 4

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [pdf] {item['titulo'][:90]}")
    conteudo = baixar_pdf(item["url"])
    if not conteudo:
        print("    -> download falhou.")
        continue
    texto = extrair_texto_pdf(conteudo)
    detalhes.append({**item, "texto": texto, "tamanho_bytes": len(conteudo)})
    print(f"    -> {len(conteudo)} bytes, {len(texto)} chars extraídos.")

print(f"\n{len(detalhes)}/{AMOSTRA} PDFs abertos com sucesso.")
curtos = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars: {len(curtos)}")

In [0]:
# Amostra completa do primeiro PDF — pra conferir na mão se bate com o
# que aparece no documento.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {detalhe['tamanho_bytes']} bytes / {len(detalhe['texto'])} chars extraídos")
print("=" * 100)
print(detalhe["texto"][:2000])

## Conclusão da Fase 1

Os dois testes passam: o `GET` simples na página (com `curl_cffi`
impersonation pra passar do WAF) já devolve os PDFs da janela recente
prontos no HTML, sem precisar do endpoint AJAX; download + extração de
texto via `pypdf` funcionam normalmente nos PDFs de exemplo.

**Avaliação para a Fase 2**: encaixa no `ingest-PDF.ipynb` (padrão
EPE-SEGOV/MS, Resoluções Agesan-RS), não no `ingest-scraping.ipynb` --
listagem aponta pra PDFs, sem texto corrido em HTML. Diferente das outras
3 fontes desse dispatcher (que usam o extrator genérico
`extrair_links_pdf`), a CCEE precisa de um `listar_ccee()` próprio pra
pegar o título específico (`.card-subtitle`, com número/data da reunião)
e a data (`.card-published`) em vez do texto genérico do link -- vai
precisar de um pequeno ajuste no dispatcher pra aceitar uma função de
listagem por fonte (mesmo padrão que o `ingest-scraping.ipynb` já usa) em
vez do `extrair_links_pdf` fixo, e passar `published_at` adiante pros
metadados (hoje sempre `None` nesse dispatcher).

**Sobre o registro**: já existe uma entrada "CCEE" no catálogo (source_id
"—", nunca implementada) -- UPDATE nela na Fase 3, sem criar linha nova.